# Encoder–HistGB: cosine annealing parameter optimization

Based on **cosine_annealing.ipynb**, keeping the encoder architecture, HistGB parameters
and shared preprocessing unchanged. Search only the initial learning rate, weight decay,
dropout and restart period.

Each fit uses an internal 20% patient holdout to select the best epoch count. Then fit
fresh preprocessing/scalers and a fresh encoder on all fold-training patients for that
fixed number of epochs, without early stopping. HistGB
trains on all fold-training patients; the fold-validation patients train neither model.

Search uses five fixed patient folds and seed 42. Confirm the two best completed trials
on the same five folds with seeds 42, 43 and 44; rank configurations by mean pipeline MSE
across all 15 runs, not by their best seed. These reused folds provide tuning/stability
scores, not an independent test estimate.


## 1. Reuse the shared baseline and set training options

Only the five setup cells from features_selection.ipynb are executed, not its validation or
feature-selection experiments. Paths work from the project root or either notebook folder.
Architecture and HistGB constants are the saved winners from opt.ipynb and histgb_opt.ipynb.
N_TRIALS controls the search budget; TOP_K=2 controls confirmation.


In [ ]:
import json
import os
from pathlib import Path
from copy import deepcopy
import optuna
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold

PROJECT_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / "exploration/features_selection.ipynb").is_file())
reference = json.loads((PROJECT_DIR / "exploration/features_selection.ipynb").read_text())
previous_dir = Path.cwd()
try:
    os.chdir(PROJECT_DIR)
    for i in [1, 3, 4, 6, 8]:  # Imports/data, history, imputation, features, pipeline.
        exec("".join(reference["cells"][i]["source"]))
finally:
    os.chdir(previous_dir)
DATA_DIR = PROJECT_DIR / "data"
baseline = clone(pipeline)

SEED = 42
CONFIRMATION_SEEDS = [42, 43, 44]
N_TRIALS, TOP_K = 30, 2
# Saved confirmed encoder winner from opt.ipynb.
ENCODER = dict(width=32, heads=8, layers=3, dropout=0.1347886862599206,
               ff_multiplier=3, mlp_layers=2, mlp_units=16)
MAX_EPOCHS, PATIENCE, BATCH_SIZE = 160, 25, 128
LEARNING_RATE, WEIGHT_DECAY = 0.001759366313629433, 4.3514588969246546e-05
MIN_LEARNING_RATE = LEARNING_RATE * 0.01
RESTART_PERIOD, RESTART_MULTIPLIER = 20, 1  # Constant 20-epoch cosine cycles.
# Saved best HistGB parameters from histgb_opt.ipynb; used for both comparisons.
HISTGB_PARAMS = {
    "learning_rate": 0.03471523676107423, "max_iter": 800,
    "max_leaf_nodes": 62, "min_samples_leaf": 16,
    "l2_regularization": 8.305016398991823, "max_features": 0.6408427464001498,
}
baseline[-1].set_params(random_state=SEED, **HISTGB_PARAMS)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")
torch.set_num_threads(3)
print(f"Device: {DEVICE}; baseline: {baseline[-1]}")

/Users/Luc/Library/CloudStorage/OneDrive-Personal/Documents/Cours/MS2A/Apprentissage Statistique/Méthodologie/Challenge_Parkinson_disease/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps; baseline: HistGradientBoostingRegressor(l2_regularization=8.305016398991823,
                              learning_rate=0.03471523676107423,
                              max_features=0.6408427464001498, max_iter=800,
                              max_leaf_nodes=62, min_samples_leaf=16,
                              random_state=42)


## 2. One sequence per patient

Patient IDs only group visits; they are never encoder inputs. Elapsed time encodes the actual
spacing between visits. Padding uses row number -1, which is ignored by attention and MSE.
The row numbers also restore embeddings to the original visit order after batching.

In [2]:
def encoder_features(imputed):
    features = imputed.drop(columns="patient_id").copy()
    features["time_since_first_visit"] = (
        imputed["age"] - imputed.groupby("patient_id")["age"].transform("min")
    )
    return features


def patient_sequences(raw, features, targets=None):
    visits = raw.reset_index(drop=True).sort_values(["patient_id", "age"])
    rows = [torch.tensor(group.index.to_numpy())
            for _, group in visits.groupby("patient_id", sort=False)]
    values = torch.tensor(np.asarray(features), dtype=torch.float32)
    labels = torch.tensor(np.zeros(len(raw)) if targets is None else np.asarray(targets),
                          dtype=torch.float32)
    return TensorDataset(
        pad_sequence([values[i] for i in rows], batch_first=True),
        pad_sequence([labels[i] for i in rows], batch_first=True),
        pad_sequence(rows, batch_first=True, padding_value=-1),
    )

## 3. Optimized bidirectional encoder

Dense input projection → three TransformerEncoder layers → one embedding per visit.
Use the same parameterized architecture as opt.ipynb: width 32, eight heads, feed-forward
multiplier three, and a regression head with two hidden layers of 16 units. The regression
head supervises training; HistGB receives the contextual embeddings.


In [3]:
class VisitEncoder(nn.Module):
    def __init__(self, n_features, width=64, heads=4, layers=2, dropout=0.1,
                 ff_multiplier=2, mlp_layers=0, mlp_units=32):
        super().__init__()
        self.projection = nn.Linear(n_features, width)
        layer = nn.TransformerEncoderLayer(
            d_model=width, nhead=heads, dim_feedforward=ff_multiplier * width,
            dropout=dropout, activation="gelu", batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers,
                                             enable_nested_tensor=False)
        # 1. Initialize the encoder’s weights
        # TransformerEncoder copies its layer: give each block independent weights.
        for parameter in self.encoder.parameters():
            if parameter.ndim > 1:
                nn.init.xavier_uniform_(parameter)
        # 2. Build the regression head
        head, input_dim = [], width
        for _ in range(mlp_layers):
            head.extend([nn.Linear(input_dim, mlp_units), nn.GELU(), nn.Dropout(dropout)])
            input_dim = mlp_units
        self.head = nn.Sequential(*head, nn.Linear(input_dim, 1))

    def encode(self, values, padding):
        return self.encoder(self.projection(values), src_key_padding_mask=padding,
                            is_causal=False)

    def forward(self, values, padding):
        return self.head(self.encode(values, padding)).squeeze(-1)

## 4. Parameterized cosine annealing and internal early stopping

AdamW uses the trial's learning rate and weight decay. CosineAnnealingWarmRestarts uses its
restart period, T_mult=1 and eta_min=1% of that trial's initial learning rate. The trial's
dropout is applied to the encoder and regression head; other architecture settings stay fixed.

Keep the 150-epoch ceiling. Patience is max(25, 2 × restart period), allowing at least two
cycles without improvement before stopping. Select the epoch with the lowest internal
holdout MSE, then refit preprocessing/scalers and a fresh encoder on all fold-training
patients for exactly that many epochs. Outer-validation targets never select the epoch.


In [4]:
def train_network(training, seed, epochs, validation=None, params=None):
    params = {"learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
              "dropout": ENCODER["dropout"], "restart_period": RESTART_PERIOD, **(params or {})}
    patience = max(PATIENCE, 2 * params["restart_period"])
    torch.manual_seed(seed)
    network = VisitEncoder(training.tensors[0].shape[-1], **{**ENCODER, "dropout": params["dropout"]}).to(DEVICE)
    optimizer = torch.optim.AdamW(network.parameters(), lr=params["learning_rate"],
                                 weight_decay=params["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=params["restart_period"], T_mult=RESTART_MULTIPLIER,
        eta_min=params["learning_rate"] * 0.01,
    )
    loader = DataLoader(training, batch_size=BATCH_SIZE, shuffle=True)
    best_loss, best_epoch, best_state, stale = np.inf, epochs, None, 0
    history = []

    for epoch in range(1, epochs + 1):
        network.train()
        total_error, count = 0.0, 0
        for values, targets, rows in loader:
            values, targets, rows = (t.to(DEVICE) for t in (values, targets, rows))
            valid = rows >= 0
            optimizer.zero_grad()
            prediction = network(values, ~valid)
            loss = nn.functional.mse_loss(prediction[valid], targets[valid])
            loss.backward()
            nn.utils.clip_grad_norm_(network.parameters(), 1.0)
            optimizer.step()
            total_error += loss.item() * valid.sum().item()
            count += valid.sum().item()
        history.append({"epoch": epoch, "train_MSE": total_error / count,
                        "learning_rate": optimizer.param_groups[0]["lr"]})

        if validation is not None:
            network.eval()
            total_error, count = 0.0, 0
            with torch.no_grad():
                for values, targets, rows in DataLoader(validation, batch_size=BATCH_SIZE):
                    values, targets, rows = (t.to(DEVICE) for t in (values, targets, rows))
                    valid = rows >= 0
                    prediction = network(values, ~valid)
                    total_error += ((prediction[valid] - targets[valid]) ** 2).sum().item()
                    count += valid.sum().item()
            stop_loss = total_error / count
            history[-1]["stop_MSE"] = stop_loss
            if stop_loss < best_loss:
                best_loss, best_epoch, stale = stop_loss, epoch, 0
                best_state = deepcopy(network.state_dict())
            else:
                stale += 1
            if stale >= patience:
                break
        scheduler.step()

    if best_state is not None:
        network.load_state_dict(best_state)
    return network.eval(), best_epoch, pd.DataFrame(history)


def fit_encoder(raw, targets, seed, params=None):
    fit_idx, stop_idx = next(GroupShuffleSplit(
        n_splits=1, test_size=0.2, random_state=seed
    ).split(raw, groups=raw["patient_id"]))
    fit_raw, stop_raw = raw.iloc[fit_idx], raw.iloc[stop_idx]
    inner_prep = clone(baseline[:2])
    fit_features = encoder_features(inner_prep.fit_transform(fit_raw))
    stop_features = encoder_features(inner_prep.transform(stop_raw))
    scaler = StandardScaler().fit(fit_features)
    mean = targets.iloc[fit_idx].mean()
    scale = max(targets.iloc[fit_idx].std(ddof=0), 1e-8)
    training = patient_sequences(fit_raw, scaler.transform(fit_features),
                                 (targets.iloc[fit_idx] - mean) / scale)
    stopping = patient_sequences(stop_raw, scaler.transform(stop_features),
                                 (targets.iloc[stop_idx] - mean) / scale)
    _, epochs, history = train_network(training, seed, MAX_EPOCHS, stopping, params=params)
    history[["train_MSE", "stop_MSE"]] *= scale ** 2

    # Fresh fit on all fold-training patients, using the selected fixed epoch count.
    full_prep = clone(baseline[:2])
    features = encoder_features(full_prep.fit_transform(raw))
    scaler = StandardScaler().fit(features)
    mean, scale = targets.mean(), max(targets.std(ddof=0), 1e-8)
    training = patient_sequences(raw, scaler.transform(features), (targets - mean) / scale)
    network, _, refit_history = train_network(training, seed, epochs, validation=None, params=params)
    refit_history["train_MSE"] *= scale ** 2

    network.requires_grad_(False)
    return dict(network=network, prep=full_prep, scaler=scaler,
                target_mean=mean, target_scale=scale,
                epochs=epochs, history=history, refit_history=refit_history)

## 5. Extract frozen embeddings in the original row order

The prediction head stays available for diagnostics; HistGB receives only the contextual
embeddings, with no CLS token, pooling or predicted-score column. Encoder inputs are
transformed using the preprocessing/scalers refitted on all fold-training patients.

In [5]:
@torch.no_grad()
def encoder_outputs(fitted, raw):
    imputed = fitted["prep"].transform(raw)
    features = fitted["scaler"].transform(encoder_features(imputed))
    sequences = patient_sequences(raw, features)
    network = fitted["network"].eval()
    embeddings = np.empty((len(raw), network.projection.out_features), dtype=np.float32)
    predictions = np.empty(len(raw), dtype=np.float32)
    for values, _, rows in DataLoader(sequences, batch_size=BATCH_SIZE):
        padding = rows < 0
        hidden = network.encode(values.to(DEVICE), padding.to(DEVICE))
        prediction = network.head(hidden).squeeze(-1).cpu()
        valid_rows = rows[~padding].numpy()
        embeddings[valid_rows] = hidden.cpu()[~padding].numpy()
        predictions[valid_rows] = prediction[~padding].numpy()
    embeddings = pd.DataFrame(embeddings, index=raw.index,
                              columns=[f"enc_{i:02d}" for i in range(embeddings.shape[1])])
    predictions = predictions * fitted["target_scale"] + fitted["target_mean"]
    return embeddings, pd.Series(predictions, index=raw.index)

## 6.1 Prepare five patient folds and evaluate the complete pipeline

Cache HistGB preprocessing once per fold. Encoder preprocessing and its patient holdout
remain inside each fit and seed. For every configuration, select the epoch count on the
internal holdout, refit a fresh encoder on all fold-training patients, extract embeddings
and train HistGB. No outer-validation label selects an epoch.


In [6]:
cv = GroupKFold(n_splits=5, shuffle=True, random_state=SEED)
prepared_folds = []
with threadpool_limits(limits=3):
    for train_idx, val_idx in cv.split(X, y, groups=X["patient_id"]):
        raw_train, raw_val = X.iloc[train_idx], X.iloc[val_idx]
        yt, yv = y.iloc[train_idx], y.iloc[val_idx]
        prep = clone(baseline[:-1])
        prepared_folds.append(dict(
            raw_train=raw_train, raw_val=raw_val, yt=yt, yv=yv,
            train=prep.fit_transform(raw_train, yt), validation=prep.transform(raw_val),
        ))


def evaluate_config(params, folds, seeds, trial=None):
    scores = []
    with threadpool_limits(limits=3):
        for fold_number, fold in enumerate(folds, 1):
            for seed in seeds:
                fitted = fit_encoder(fold["raw_train"], fold["yt"], seed, params=params)
                z_train, _ = encoder_outputs(fitted, fold["raw_train"])
                z_val, _ = encoder_outputs(fitted, fold["raw_val"])
                model = clone(baseline[-1]).set_params(random_state=seed)
                model.fit(fold["train"].join(z_train), fold["yt"])
                mse = mean_squared_error(fold["yv"], model.predict(fold["validation"].join(z_val)))
                scores.append(dict(fold=fold_number, seed=seed, epochs=fitted["epochs"], MSE=mse))
                print(f"Fold {fold_number}/5, seed {seed}, {fitted['epochs']} epochs: MSE {mse:.4f}")
            if trial is not None:
                trial.report(float(np.mean([s["MSE"] for s in scores])), step=fold_number)
                if trial.should_prune():
                    raise optuna.TrialPruned()
    return pd.DataFrame(scores)


## 6.2 Optuna search: five folds, one seed

Search 30 configurations by default, starting with the existing settings. Learning rate:
1e-4–5e-3; weight decay: 1e-6–1e-2 (both logarithmic); dropout: 0–0.35; restart period:
10, 15, 20, 30 or 40 epochs. Minimize complete Encoder–HistGB MSE. Median pruning can
stop trials after at least two folds, following five startup trials.


In [7]:
def objective(trial):
    params = dict(
        learning_rate=trial.suggest_float("learning_rate", 1e-4, 5e-3, log=True),
        weight_decay=trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        dropout=trial.suggest_float("dropout", 0.0, 0.35),
        restart_period=trial.suggest_categorical("restart_period", [10, 15, 20, 30, 40]),
    )
    result = evaluate_config(params, prepared_folds, [SEED], trial)
    trial.set_user_attr("fold_scores", result.to_dict("records"))
    return float(result["MSE"].mean())


study = optuna.create_study(
    direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),
)
study.enqueue_trial(dict(learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
                         dropout=ENCODER["dropout"], restart_period=RESTART_PERIOD))
study.optimize(objective, n_trials=N_TRIALS, gc_after_trial=True, show_progress_bar=True)
study.trials_dataframe().sort_values("value")


[I 2026-11-03 15:07:27,064] A new study created in memory with name: no-name-dc7d06cd-cc6c-4d7b-90d6-4b2bc44c95e9
  0%|          | 0/30 [00:00<?, ?it/s]

Fold 1/5, seed 42, 133 epochs: MSE 7.3291
Fold 2/5, seed 42, 140 epochs: MSE 7.2668
Fold 3/5, seed 42, 139 epochs: MSE 7.4121
Fold 4/5, seed 42, 136 epochs: MSE 7.5657


Best trial: 0. Best value: 7.3327:   3%|▎         | 1/30 [16:12<7:50:10, 972.78s/it]

Fold 5/5, seed 42, 140 epochs: MSE 7.0898
[I 2026-11-03 15:23:39,764] Trial 0 finished with value: 7.332704580022731 and parameters: {'learning_rate': 0.001759366313629433, 'weight_decay': 4.3514588969246546e-05, 'dropout': 0.1347886862599206, 'restart_period': 20}. Best is trial 0 with value: 7.332704580022731.
Fold 1/5, seed 42, 145 epochs: MSE 8.2086
Fold 2/5, seed 42, 149 epochs: MSE 8.4732
Fold 3/5, seed 42, 150 epochs: MSE 8.5054
Fold 4/5, seed 42, 147 epochs: MSE 8.6742


Best trial: 0. Best value: 7.3327:   7%|▋         | 2/30 [32:00<7:27:09, 958.21s/it]

Fold 5/5, seed 42, 148 epochs: MSE 8.5066
[I 2026-11-03 15:39:27,790] Trial 1 finished with value: 8.473598108910853 and parameters: {'learning_rate': 0.00043284502212938834, 'weight_decay': 0.006351221010640699, 'dropout': 0.25619787963399177, 'restart_period': 40}. Best is trial 0 with value: 7.332704580022731.
Fold 1/5, seed 42, 98 epochs: MSE 7.9778
Fold 2/5, seed 42, 119 epochs: MSE 7.7391
Fold 3/5, seed 42, 107 epochs: MSE 7.8630
Fold 4/5, seed 42, 127 epochs: MSE 8.4010


... (53 outputs left out)

Fold 2/5, seed 42, 77 epochs: MSE 7.3801
[I 2026-11-03 19:06:13,186] Trial 28 pruned. 
Fold 1/5, seed 42, 150 epochs: MSE 7.2681


Best trial: 23. Best value: 7.03532: 100%|██████████| 30/30 [4:02:48<00:00, 485.63s/it]

Fold 2/5, seed 42, 149 epochs: MSE 7.4593
[I 2026-11-03 19:10:15,908] Trial 29 pruned. 


,number,value,datetime_start,datetime_complete,duration,params_dropout,params_learning_rate,params_restart_period,params_weight_decay,user_attrs_fold_scores,system_attrs_fixed_params,system_attrs_tpe:relative_params:0,state
23,23,7.035321,2026-11-03 18:16:52.790207,2026-11-03 18:25:31.671348,0 days 00:08:38.881141,0.191817,0.004607,40,0.002606,"[{'fold': 1, 'seed': 42, 'epochs': 117, 'MSE':...",NaN,"{""dropout"": 0.19181651819990855, ""learning_rat...",COMPLETE
19,19,7.088271,2026-11-03 17:47:32.002033,2026-11-03 17:55:57.226252,0 days 00:08:25.224219,0.206951,0.004865,40,0.000253,"[{'fold': 1, 'seed': 42, 'epochs': 79, 'MSE': ...",NaN,"{""dropout"": 0.20695064464625137, ""learning_rat...",COMPLETE
22,22,7.091146,2026-11-03 18:08:13.225877,2026-11-03 18:16:52.735833,0 days 00:08:39.509956,0.197761,0.003892,40,0.004475,"[{'fold': 1, 'seed': 42, 'epochs': 117, 'MSE':...",NaN,"{""dropout"": 0.1977613302607443, ""learning_rate...",COMPLETE
21,21,7.135883,2026-11-03 17:59:44.068717,2026-11-03 18:08:13.174746,0 days 00:08:29.106029,0.221339,0.003819,40,0.003832,"[{'fold': 1, 'seed': 42, 'epochs': 117, 'MSE':...",NaN,"{""dropout"": 0.2213393097245546, ""learning_rate...",COMPLETE
27,27,7.154090,2026-11-03 18:54:56.235403,2026-11-03 19:03:12.380304,0 days 00:08:16.144901,0.139074,0.004353,40,0.004491,"[{'fold': 1, 'seed': 42, 'epochs': 116, 'MSE':...",NaN,"{""dropout"": 0.13907433546959863, ""learning_rat...",COMPLETE
14,14,7.177798,2026-11-03 17:03:58.366310,2026-11-03 17:12:54.409365,0 days 00:08:56.043055,0.225101,0.003795,40,0.000095,"[{'fold': 1, 'seed': 42, 'epochs': 117, 'MSE':...",NaN,"{""dropout"": 0.22510089416402457, ""learning_rat...",COMPLETE
17,17,7.202395,2026-11-03 17:29:45.033149,2026-11-03 17:38:10.598068,0 days 00:08:25.564919,0.172593,0.004091,40,0.000016,"[{'fold': 1, 'seed': 42, 'epochs': 118, 'MSE':...",NaN,"{""dropout"": 0.17259277421518518, ""learning_rat...",COMPLETE
24,24,7.235209,2026-11-03 18:25:31.719394,2026-11-03 18:35:05.305542,0 days 00:09:33.586148,0.176067,0.002815,40,0.001841,"[{'fold': 1, 'seed': 42, 'epochs': 147, 'MSE':...",NaN,"{""dropout"": 0.17606706086036017, ""learning_rat...",COMPLETE
12,12,7.240413,2026-11-03 16:47:23.254627,2026-11-03 16:56:30.767553,0 days 00:09:07.512926,0.147242,0.002453,20,0.000099,"[{'fold': 1, 'seed': 42, 'epochs': 133, 'MSE':...",NaN,"{""dropout"": 0.14724173374074423, ""learning_rat...",COMPLETE
16,16,7.242883,2026-11-03 17:22:19.681621,2026-11-03 17:29:44.982496,0 days 00:07:25.300875,0.123207,0.004504,40,0.000608,"[{'fold': 1, 'seed': 42, 'epochs': 116, 'MSE':...",NaN,"{""dropout"": 0.1232069191610845, ""learning_rate...",COMPLETE


## 6.3 Confirm the best two: five folds, three seeds

Retrain each of the two best completed configurations on all five folds with seeds 42, 43
and 44. Each seed controls encoder initialization, the internal patient holdout and HistGB.
Select the epoch count on the internal holdout, then refit on all fold-training patients. The winning
configuration minimizes mean MSE across its 15 runs. Standard deviations describe
fold/seed variation; the patients and validation folds are reused from model selection.


In [8]:
candidates = sorted(
    (t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE),
    key=lambda t: t.value,
)[:TOP_K]
if len(candidates) < TOP_K:
    raise ValueError("At least two completed trials are required for confirmation.")

confirmation = []
for candidate in candidates:
    print(f"Confirming trial {candidate.number}: {candidate.params}")
    result = evaluate_config(candidate.params, prepared_folds, CONFIRMATION_SEEDS)
    confirmation.append(result.assign(trial=candidate.number))
confirmation = pd.concat(confirmation, ignore_index=True)
confirmation_summary = confirmation.groupby("trial").agg(
    MSE=("MSE", "mean"), std=("MSE", lambda s: s.std(ddof=0)),
    mean_epochs=("epochs", "mean"),
).sort_values("MSE")
confirmation_summary["RMSE"] = np.sqrt(confirmation_summary["MSE"])
winning_trial = int(confirmation_summary.index[0])
winning_params = next(t.params for t in candidates if t.number == winning_trial)
print("Winning parameters:", winning_params)
confirmation_summary


Confirming trial 23: {'learning_rate': 0.0046066691407564055, 'weight_decay': 0.0026062405166073018, 'dropout': 0.19181651819990855, 'restart_period': 40}
Fold 1/5, seed 42, 117 epochs: MSE 6.7231
Fold 1/5, seed 43, 150 epochs: MSE 6.8771
Fold 1/5, seed 44, 116 epochs: MSE 6.7926
Fold 2/5, seed 42, 117 epochs: MSE 7.1139
Fold 2/5, seed 43, 117 epochs: MSE 7.1350
Fold 2/5, seed 44, 114 epochs: MSE 7.2267
Fold 3/5, seed 42, 116 epochs: MSE 7.0111
Fold 3/5, seed 43, 148 epochs: MSE 7.0424
Fold 3/5, seed 44, 146 epochs: MSE 7.2774
Fold 4/5, seed 42, 139 epochs: MSE 7.4515
Fold 4/5, seed 43, 117 epochs: MSE 7.4286
Fold 4/5, seed 44, 150 epochs: MSE 7.5135
Fold 5/5, seed 42, 116 epochs: MSE 6.8772
Fold 5/5, seed 43, 117 epochs: MSE 7.2755
Fold 5/5, seed 44, 120 epochs: MSE 7.0894
Confirming trial 19: {'learning_rate': 0.004865350690227582, 'weight_decay': 0.0002525551694486565, 'dropout': 0.20695064464625137, 'restart_period': 40}
Fold 1/5, seed 42, 79 epochs: MSE 6.9597
Fold 1/5, seed 43, 1

,MSE,std,mean_epochs,RMSE
trial,,,,
23,7.122336,0.234102,126.666667,2.668771
19,7.189356,0.209531,126.200000,2.681297
